
# Economic model and HJB equations


## Model specification <a id="model-spec"></a>

### State variables

| Symbol | Description | Stochastic Differential Equation |
|---|---|---|
| $\hat{k}_t = \log(K_t)$ | Productive capital | $\displaystyle dK_t = K_t\Big[ -\mu_k + \frac{I^k_t}{K_t} - \frac{\kappa}{2}\Big(\frac{I^k_t}{K_t}\Big)^2 \Big]\,dt + K_t\,\sigma_k\,dW_t$ |
| $Y_t$ | Temperature anomaly | $\displaystyle dY_t = \mathcal{E}_t\big[\theta\,dt + \varsigma\,dW_t\big]$ |
| $\hat{r}_t = \log(R_t)$ | R&D capital | $\displaystyle dR_t = -\zeta R_t\,dt + \psi_0(I^r_t)^{\psi_1}(R_t)^{1-\psi_1}\,dt + R_t\,\sigma_r\,dW_t$ |

$W_t$ is a standard Brownian motion. 

### Damage function <a href="https://github.com/korito1416/two-capital-climate-change/blob/main/python/frequency_plot.py" class="code-link" target="_blank" rel="noopener"></a>
$$
\frac{d \hat{n}}{dy} =
\begin{cases}
\lambda_1 + \lambda_2 y, & \text{if } y \le \hat{y} \\
\lambda_1 + \lambda_2 \left( y - \hat{y} + \overline{y} \right) + \lambda_3(\ell)(y - \hat{y}), & \text{if } y > \hat{y}
\end{cases}
$$

By integration, we get

$$
\hat{n}(y) = 
\begin{cases}
\lambda_1 y + \dfrac{\lambda_2}{2} y^2, & \text{if } 0 \le y \le \hat{y} \\
\lambda_1 y + \dfrac{\lambda_2}{2} \hat{y}^2 + 
\dfrac{\lambda_2}{2} (y - \hat{y} + \overline{y})^2 + 
\dfrac{\lambda_3(\ell)}{2} (y - \hat{y})^2 - 
\dfrac{\lambda_2}{2} \overline{y}^2, & \text{if } y \ge \hat{y}
\end{cases}
$$


<iframe
  id="plotly-div"
  src="../_static/damage_interactive.html"
  width="100%"
  height="470"
  style="border:none;"
></iframe>


<!-- ——— Reproduction instructions ——— -->
<div style="margin:0.8rem 0 0.2rem;">
To reproduce Figure 1, Figure 2, and Figure 8 in the paper, run the shell scripts provided below.
</div>

~~~~bash
bash ./sbatch/baseline_plot.sh
~~~~

After the run finishes, you should have output files named
`Climate_Sensitivity.png`, `Temperature Anomaly.png`, and `Temperature Anomaly weighted.png` under `output/figures/<dataset>/baseline_plot/`.

Source scripts: 
* [**`baseline_plot.sh`**](https://github.com/korito1416/two-capital-climate-change/blob/main/sbatch/baseline_plot.sh)


### Intensity function
$$
\mathcal{J}^n(y) = 
\begin{cases}
0, & \text{if } 0 \le y \le \underline{y} \\
\mathsf{d}_0 \exp\left[ 
\frac{\mathsf{d}_1}{2} (y - \underline{y})^2 + 
\frac{\mathsf{d}_2}{2} (y - \overline{y})^{-2}
\right] - 
\mathsf{d}_0 \exp\left[ 
\frac{\mathsf{d}_2}{2} (\underline{y} - \overline{y})^{-2}
\right], & \text{if } \underline{y} < y < \overline{y}
\end{cases}
$$

<iframe
  id="plotly-div"
  src="../_static/intensity_interactive.html"
  width="80%"
  height="420"
  style="display:block; margin:0 auto; border:none;"
></iframe>

Given that a damage information event takes place, each of the $L-1$ damage curves is realized with equal probability, so curve $\ell$ has intensity

$$
\mathcal{J}^{\ell}(y) = \frac{1}{L-1}\,\mathcal{J}^n(y), \qquad \ell = 1, 2, \dots, L-1.
$$

The technology jump intensity is proportional to the stock of knowledge,

$$
\mathcal{J}^{L}(R_t) = \chi R_t
$$

### Controls
| Symbol | Description |
|---|---|
| $I^k_t$ | Investment in productive capital   |
| $I^r_t$ | Investment in R&D |
| $\mathcal{E}_t$ | Emission |

### Misspecifications

| Symbol | Channel | Interpretation |
|---|---|---|
| $h_k$ | Capital | Brownian motion misspecification of $K_t$ |
| $h_r$ | Knowledge | Brownian motion misspecification of $R_t$ |
| $q(\theta)$ | Climate | Robust averaging over the 144 climate models (TCREs $\theta \in \Theta$) |
| $\{ g^\ell \}_{\ell = 1}^{L-1}$ | Damage jump | Jump misspecification of damage‑jump arrival |
| $g^L$ | Tech jump | Jump misspecification of technology‑jump arrival |


Robustness is governed by two penalty parameters: $\xi_m$, shared by the Brownian and jump misspecification channels, and ${\xi_a} $ for the robust averaging across climate models.



## Hamilton–Jacobi–Bellman Equations <a id="hjb"></a>

We distinguish four regimes:

| Case                         | Jump History        | State Vector \(x\)                        | Controls                       | Misspecification                          |
|------------------------------|---------------------|-------------------------------------------|--------------------------------|-------------------------------------------|
| (i) Post-Tech, Post-Damage  | tech ✓, damage ✓    | $\{\hat{k},\, z\}$                        | $\{i_k\}$                      | $\{h_k\}$                                 |
| (ii) Post-Tech, Pre-Damage  | tech ✓, damage ✗    | $\{\hat{k}\}$                             | $\{i_k\}$                      | $\{h_k\}$                                 |
| (iii) Pre-Tech, Post-Damage | tech ✗, damage ✓    | $\{\hat{k},\, z,\, \hat{r}\}$             | $\{i_k,\, i_r,\, e\}$          | $\{h_k,\, h_r,\, q(\theta),\, g^L\}$              |
| (iv) Pre-Tech, Pre-Damage   | tech ✗, damage ✗    | $\{\hat{k},\, y,\, \hat{r}\}$             | $\{i_k,\, i_r,\, e\}$          | $\{h_k,\, h_r,\, q(\theta),\, g^L,\, \{ g^\ell \}_{\ell = 1}^{L-1}\}$         |

In the HJB equations below, $i^k$ and $i^r$ denote the investment-capital ratios $I^k/K$ and $I^r/K$.

Define 

$$
\begin{align*}
    z &\overset{\mathrm{def}}{=} y - {\hat y} + {\bar y}\\
   \hat{m}(z;\ell) &\overset{\mathrm{def}}{=} {\hat n}(y) - (\lambda_1{\hat y}+ \frac 1 2 \lambda_2 {\hat y} ^2)\\
   W^{\ell,L}(\hat{k},z) &\overset{\mathrm{def}}{=} V^{\ell,L,\hat{y}}(\hat{k},y) + (\lambda_1{\hat y}+ \frac 1 2 \lambda_2 {\hat y} ^2)\\
   W^{\ell}(\hat{k},z,\hat{r}) &\overset{\mathrm{def}}{=} V^{\ell,\hat{y}}(\hat{k},y,\hat{r}) + (\lambda_1{\hat y}+ \frac 1 2 \lambda_2 {\hat y} ^2)
\end{align*}
$$

Then

$$
\begin{align*}
   \hat{m}(z;\ell) &= \lambda_1(y - \hat{y})+{\frac 1 2}\lambda_2 (y - {\hat y} + {\bar y} )^2 + {\frac 1 2}\lambda_3(\ell) (y- \hat y)^2 - {\frac 1 2}\lambda_2(\bar y)^2\\
   &= \lambda_1(z-\bar y) +{\frac 1 2}\lambda_2z^2+{\frac 1 2}\lambda_3(\ell)(z-\bar y)^2- {\frac 1 2}\lambda_2(\bar y)^2
\end{align*}
$$

### Post-Tech, Post-Damage
We can get $W^{\ell,L}(\hat{k},z)$ from

$$
\begin{align*}
0 = & \max_{i^k}\min_{h^k}\ \delta\log(\alpha - i^k) + \delta\hat{k} - \delta\hat{m}(\cdot;\ell) - \delta W^{\ell,L} + \xi_m\frac{\left|h^k\right|^2}{2} \\
& + \frac{\partial W^{\ell,L}}{\partial\hat{k}}\left[-\mu_k + i^k - \frac{\kappa}{2}\left(i^k\right)^2 - \frac{|\sigma_k|^2}{2} + \sigma_k h\right] + \frac{\partial^2 W^{\ell,L}}{\partial\hat{k}^2}\frac{|\sigma_k|^2}{2}
\end{align*}
$$

### Post-Tech, Pre-Damage
We can get $V^{L}(\hat{k})$ from

$$
\begin{align*}
0 = & \max_{i^k}\min_{h^k}\ \delta\log(\alpha - i^k) + \delta\hat{k} - \delta\hat{n}(\cdot;\ell) - \delta V^L + \xi_m\frac{\left|h^k\right|^2}{2} \\
& + \frac{\partial V^L}{\partial\hat{k}}\left[-\mu_k + i^k - \frac{\kappa}{2}\left(i^k\right)^2 - \frac{|\sigma_k|^2}{2} + \sigma_k h\right] + \frac{\partial^2 V^L}{\partial\hat{k}^2}\frac{|\sigma_k|^2}{2},
\end{align*}
$$

It can be solved with the same guess-and-verify argument as the previous regime.

### Pre-Tech, Post-Damage
From Post-Post we know $W^{\ell,L}(\hat{k},z)$, then we can get $W^{\ell}(\hat{k},z,\hat{r})$ from

$$
\begin{align*}
0 = & \max_{i^k, i^r, e}\ \min_{h, q(\theta), g^L}\ \delta\log\left[\alpha - i^k - i^r - \alpha\phi_0\left(1 - \frac{e}{\beta\alpha k}\right)^{\phi_1}\right] + \delta\hat{k} - \delta\hat{m}(z;\ell) - \delta W^\ell(\hat{k}, z, \hat{r}) \\
& + \frac{\partial W^\ell}{\partial\hat{k}}(\hat{k}, z, \hat{r})\left[-\mu_k + i^k - \frac{\kappa}{2}\left(i^k\right)^2 - \frac{|\sigma_k|^2}{2} + \sigma_k h\right] + \frac{\partial^2 W^\ell}{\partial\hat{k}\,\partial\hat{k}'}(\hat{k}, z, \hat{r})\frac{|\sigma_k|^2}{2} \\
& + \frac{\partial W^\ell}{\partial y}(\hat{k}, z, \hat{r})\,e\left[\sum_{\theta\in\Theta}\theta q(\theta)\right] + \frac{\partial^2 W^\ell}{\partial y\,\partial y'}(\hat{k}, z, \hat{r})\frac{|\varsigma|^2}{2}e^2 \\
& + \frac{\partial W^\ell}{\partial\hat{r}}(\hat{k}, z, \hat{r})\left[-\zeta + \psi_0 (i^r)^{\psi_1}\exp\left(-\psi_1\left(\hat{r} - \hat{k}\right)\right) - \frac{|\sigma_r|^2}{2} + \sigma_r h\right] + \frac{\partial^2 W^\ell}{\partial\hat{r}\,\partial\hat{r}'}(\hat{k}, z, \hat{r})\frac{|\sigma_r|^2}{2} \\
& + \mathcal{J}^L g^L\left[W^{\ell,L}(\hat{k}, z, \hat{r}) - W^\ell(\hat{k}, z, \hat{r})\right] + \xi_m\mathcal{J}^L\left[1 - g^L + g^L\log g^L\right] + \frac{\xi_m}{2}h'h + \xi_a\sum_{\theta\in\Theta} q(\theta)\log\left(\frac{q(\theta)}{p(\theta)}\right).
\end{align*}
$$

As a second step, the post-damage, pre-technology value function follows as

$$
V^{\ell,\hat{y}}(\hat{k},y,\hat{r}) = W^{\ell}(\hat{k},\, y-\hat{y}+\bar{y},\, \hat{r}) - \left(\lambda_1{\hat y}+ \frac 1 2 \lambda_2 {\hat y}^2\right).
$$

### Pre-Tech, Pre-Damage
The inputs are the post-jump value functions: $V^{\ell,\hat{y}}$ for $1 \le \ell \le L-1$ (from Pre-Tech, Post-Damage) and $V^{L}$ (from Post-Tech, Pre-Damage). Prior to any jump realization, the recursive equation compares the current continuation value with what would happen hypothetically if a jump occurred at the current temperature anomaly $y$, so we only need $V^{\ell,\hat{y}}$ evaluated at $\hat{y} = y$. Define

$$
V^{\ell} \overset{\mathrm{def}}{=} V^{\ell,\hat{y}}\big|_{\hat{y}=y}, \qquad \ell = 1,\dots,L-1.
$$

The complete HJB equation for $V(\hat{k},y,\hat{r})$, with $V^{L}$ entering as the $\ell = L$ term, is

$$
\begin{align*}
0 = & \max_{i^k, i^r, e}\ \min_{h, q(\theta), g^\ell, g^L}\ \delta\log\left[\alpha - i^k - i^r - \alpha\phi_0\left(1 - \frac{e}{\beta\alpha k}\right)^{\phi_1}\right] + \delta\hat{k} - \delta\hat{n}(y) - \delta V(\hat{k}, y, \hat{r}) \\
& + \frac{\partial V}{\partial\hat{k}}(\hat{k}, y, \hat{r})\left[-\mu_k + i^k - \frac{\kappa}{2}\left(i^k\right)^2 - \frac{|\sigma_k|^2}{2} + \sigma_k h\right] + \frac{\partial^2 V}{\partial\hat{k}\,\partial\hat{k}'}(\hat{k}, y, \hat{r})\frac{|\sigma_k|^2}{2} \\
& + \frac{\partial V}{\partial y}(\hat{k}, y, \hat{r})\,e\left[\sum_{\theta\in\Theta}\theta q(\theta)\right] + \frac{\partial^2 V}{\partial y\,\partial y'}(\hat{k}, y, \hat{r})\frac{|\varsigma|^2}{2}e^2 \\
& + \frac{\partial V}{\partial\hat{r}}(\hat{k}, y, \hat{r})\left[-\zeta + \psi_0 (i^r)^{\psi_1}\exp\left(-\psi_1\left(\hat{r} - \hat{k}\right)\right) - \frac{|\sigma_r|^2}{2} + \sigma_r h\right] + \frac{\partial^2 V}{\partial\hat{r}\,\partial\hat{r}'}(\hat{k}, y, \hat{r})\frac{|\sigma_r|^2}{2} \\
& + \sum_{\ell=1}^L \mathcal{J}^\ell g^\ell\left[V^\ell(\hat{k}, y, \hat{r}) - V(\hat{k}, y, \hat{r})\right] + \xi_m\sum_{\ell=1}^L \mathcal{J}^\ell\left[1 - g^\ell + g^\ell\log g^\ell\right] + \frac{\xi_m}{2}h'h + \xi_a\sum_{\theta\in\Theta} q(\theta)\log\left(\frac{q(\theta)}{p(\theta)}\right).
\end{align*}
$$


## Computational method
### Policy iteration <a href="https://github.com/korito1416/two-capital-climate-change/blob/main/python/src/postsolver_new_rho1.py" class="code-link" target="_blank" rel="noopener"></a>

For simplicity, denote the state space, control set, and distortion set by  

$$
X = \{\log k,\, y,\, \log r\}, \qquad
\Phi = \{ i^k,\, i^r,\, \mathcal{E}\}, \qquad
\Gamma = \{ h^k,\, h^r,\, q(\theta),\, g^\ell,\, g^L\}.
$$

The value function $\hat{v}(X)$ depends on the state $X$.


**Input:** initial guesses $\hat{v}^0$, $\Phi^0$, $\Gamma^0$ and values for $\Delta\tau$, $\varepsilon_{relax}$, $\epsilon_{KSP}$, $\epsilon_{outer}$, $N_{max}$  
**Output:** optimal value function $\hat{v}^{*}$  

1. **Initialise** $\hat{v}\leftarrow \hat{v}^0$.  
2. **Repeat**  
   1. **Step 1 (controls).** Compute optimal actions $\Phi^{*}$ from the maximisation first-order conditions using $(\hat{v},\Phi,\Gamma)$. Update using the numerical relaxation scheme with weight $\varepsilon_{relax}$.  
   2. **Step 2 (distortions).** Compute optimal probability distortions $\Gamma^{*}$ from the minimisation first-order conditions using $(\hat{v},\Phi^{*},\Gamma)$.  
   3. **Step 3 (value update).** Obtain a provisional value $\hat{v}^{\text{new}}$ by solving (conditionally) the HJB PDE with a false-transient method, given $(\hat{v},\Phi^{*},\Gamma^{*})$.  
   4. **Step 4.** Set $\hat{v}^{prior}\leftarrow\hat{v},\;\hat{v}\leftarrow\hat{v}^{\text{new}},\;\Phi\leftarrow\Phi^{*},\;\Gamma\leftarrow\Gamma^{*}$.  
3. **Until** $\left\|\dfrac{\hat{v}^{\text{new}}-\hat{v}^{prior}}{\Delta\tau}\right\|_\infty \le \epsilon_{outer}$ or $N \ge N_{max}$.

### Updating rules for $\Phi$ and $\Gamma$ at policy-improvement steps <a href="https://github.com/korito1416/two-capital-climate-change/blob/main/python/src/presolver_crs2_new_rho1.py" class="code-link" target="_blank" rel="noopener"></a>

In solving HJB equations, we often encounter complex, highly non-linear equations for our optimal control choices that do not admit analytical solutions. To address this challenge, we implement a numerical relaxation updating scheme. The relaxation weight $\varepsilon$ determines the updating rate of the control vector $\Phi$. The control vector is updated according to

$$
\Phi_{n+1} = \varepsilon_{relax}\,\Phi^* + (1 - \varepsilon_{relax})\,\Phi_n
$$

where $\Phi_{n+1}$ is the next iteration's optimal control vector, $\Phi^*$ is the updating optimal control vector value using the current value function solution, and $\Phi_n$ is the previous iteration's optimal control vector.

Every probability distortion arising from misspecification aversion does possess a closed-form expression in terms of $\hat{v}$ and its derivatives, so those quantities are updated directly whenever $\hat{v}$ is updated.

### Solving the linear PDE at the policy-evaluation step 

To update the value function we solve a **conditionally linear** PDE implicitly. A false-transient time dimension is introduced to stabilise the non-linear HJB; implicit time-stepping continues until the artificial time derivative converges
to zero. Sparse linear systems arising at each step are solved with PETSc’s stabilised Bi-CG method and an ILU preconditioner.

The HJB can be written as  

$$
A\,\hat{v}
+ B_{\hat{k}}\,\partial_{\hat{k}}\hat{v}
+ B_{y}\,\partial_{y}\hat{v}
+ B_{\hat{r}}\,\partial_{\hat{r}}\hat{v}
+ C_{\hat{k}}\,\partial_{\hat{k}\hat{k}}\hat{v}
+ C_{y}\,\partial_{yy}\hat{v}
+ C_{\hat{r}}\,\partial_{\hat{r}\hat{r}}\hat{v}
+ D = 0.
$$

More compactly, letting $x$ collect the state variables and $V$ denote
$\hat{v}$,

$$
0 = V_t(x) +
\mathbb{A}(x;V,V_x,V_{xx})\,V +
\mathbb{B}(x;V,V_x,V_{xx})\,V_x +
\frac12 \operatorname{tr}\!\bigl[
  \mathbb{C}(x;V,V_x,V_{xx})\,
  V_{xx}\,
  \mathbb{C}(x;V,V_x,V_{xx})
\bigr]
+ \mathbb{D}(x;V,V_x,V_{xx}).
$$

Given an iterate $V^{0}(x)$ we compute the coefficients and advance via  

$$
V^{\text{new}}(x)=
V^{0}(x)+
\Bigl[
  \mathbb{A}\,V^{0}
  + \mathbb{B}\,V^{0}_x
  + \frac12\operatorname{tr}\!\bigl(\mathbb{C}\,V^{0}_{xx}\,\mathbb{C}\bigr)
  + \mathbb{D}
\Bigr]\Delta t ,
$$

then solve the resulting linear system with Bi-CG-Stab.

See [**`PETSc`**](https://github.com/korito1416/two-capital-climate-change/blob/main/README.md) for PETSc setup.

### Finite-difference scheme <a href="https://github.com/korito1416/two-capital-climate-change/blob/main/python/src/utility.py" class="code-link" target="_blank" rel="noopener"></a>

For computing the derivative terms used to evaluate the optimal controls:

* **Interior points** — central differences  

  $$
    \left(\frac{\partial f}{\partial x}\right)_i
      = \frac{f_{i+1}-f_{i-1}}{2\Delta x},
  \qquad
    \left(\frac{\partial^2 f}{\partial x^2}\right)_i
      = \frac{f_{i+1}+f_{i-1}-2f_i}{\Delta x^2}.
  $$

* **Lower boundary** — forward differences  

  $$
    \left(\frac{\partial f}{\partial x}\right)_0
      = \frac{f_{1}-f_{0}}{\Delta x},
  \qquad
    \left(\frac{\partial^2 f}{\partial x^2}\right)_0
      = \frac{f_{2}+f_{0}-2f_{1}}{\Delta x^2}.
  $$

* **Upper boundary** — backward differences  

  $$
    \left(\frac{\partial f}{\partial x}\right)_{N-1}
      = \frac{f_{N-1}-f_{N-2}}{\Delta x},
  \qquad
    \left(\frac{\partial^2 f}{\partial x^2}\right)_{N-1}
      = \frac{f_{N-1}+f_{N-3}-2f_{N-2}}{\Delta x^2}.
  $$


## Solution

Production runs are scripted:

* [**`postdamage.sh`**](https://github.com/korito1416/two-capital-climate-change/blob/main/sbatch/postdamage.sh) – solves post‑damage regimes from a cold start
* [**`postdamage_sub.sh`**](https://github.com/korito1416/two-capital-climate-change/blob/main/sbatch/postdamage_sub.sh) – continues the post‑damage solves from a converged warm start  
* [**`predamage.sh`**](https://github.com/korito1416/two-capital-climate-change/blob/main/sbatch/predamage.sh) – solves pre‑damage regimes  

For the full model, $\xi_a$ is calibrated by iteration on the Coarse grid: the post- and pre-damage solves and the prepare step are repeated (case `3`) until the change in the calibrated $\xi_a$ is below $10^{-7}$. The results reported in the paper use the Fine grid (`--grid-profile double_resolution`), which is solved next and initialized from the converged Coarse solutions. Commands without `--grid-profile double_resolution` reproduce the Coarse results, which differ slightly from the paper. The full command lists are in [`manifest_replication/master_HJB.sh`](https://github.com/korito1416/two-capital-climate-change/blob/main/manifest_replication/master_HJB.sh) and [`manifest_replication/master_post_HJB.sh`](https://github.com/korito1416/two-capital-climate-change/blob/main/manifest_replication/master_post_HJB.sh).

~~~~bash
bash ./sbatch/postdamage.sh --model full_model
bash ./sbatch/postdamage_sub.sh --model full_model calibrating_xia 1
bash ./sbatch/predamage.sh --model full_model calibrating_xia 1
bash ./sbatch/feymannkacs_prepare.sh --model full_model 1
bash ./sbatch/postdamage_sub.sh --model full_model calibrating_xia 3
bash ./sbatch/predamage.sh --model full_model calibrating_xia 3
bash ./sbatch/feymannkacs_prepare.sh --model full_model 3

bash ./sbatch/postdamage.sh --model full_model --grid-profile double_resolution --initial-guess-grid-profile baseline
bash ./sbatch/predamage.sh --model full_model --grid-profile double_resolution --initial-guess-grid-profile baseline baseline_xia
~~~~

### $\frac{\partial V}{\partial logR}$

<style>
  .btn-group {
    display: inline-flex;
    gap: 0.5em;
    margin-bottom: 1em;
  }
  .btn-group button {
    padding: 0.5em 1em;
    border: 2px solid #007acc;
    background: white;
    color: #007acc;
    font-size: 1rem;
    border-radius: 0.5em;
    cursor: pointer;
    transition: background 0.2s, color 0.2s;
  }
  .btn-group button.active,
  .btn-group button:hover {
    background: #007acc;
    color: white;
  }
  .plot-container {
    margin-top: 1em;
  }
</style>

<div class="btn-group">
  <button data-target="xik0.05" class="active">more aversion</button>
  <button data-target="xik0.1">less aversion</button>
  <button data-target="xik1e5">baseline</button>
</div>

<div id="plot-xik0.05" class="plot-container">
  <iframe src="../_static/df_V_m3_surface_xik0.05.html"
          width="100%" height="470" style="border:none;"></iframe>
</div>
<div id="plot-xik0.1" class="plot-container" style="display:none;">
  <iframe src="../_static/df_V_m3_surface_xik0.1.html"
          width="100%" height="470" style="border:none;"></iframe>
</div>
<div id="plot-xik1e5" class="plot-container" style="display:none;">
  <iframe src="../_static/df_V_m3_surface_xik1e+05.html"
          width="100%" height="470" style="border:none;"></iframe>
</div>

<script>
  const buttons = document.querySelectorAll('.btn-group button');
  buttons.forEach(btn => {
    btn.addEventListener('click', () => {
      // toggle active class
      buttons.forEach(b => b.classList.remove('active'));
      btn.classList.add('active');
      // show the right plot
      const sel = btn.getAttribute('data-target');
      ['xik0.05','xik0.1','xik1e5'].forEach(id => {
        document.getElementById('plot-'+id).style.display =
          (id === sel ? 'block' : 'none');
      });
    });
  });
</script>


### $\frac{\partial V}{\partial Y}$

<style>
  /* button bar & plot box (re‑use safely; identical rules are fine) */
  .btn-group {
    display: inline-flex;
    gap: 0.5em;
    margin-bottom: 1em;
  }
  .btn-group button {
    padding: 0.5em 1em;
    border: 2px solid #007acc;
    background: white;
    color: #007acc;
    font-size: 1rem;
    border-radius: 0.5em;
    cursor: pointer;
    transition: background 0.2s, color 0.2s;
  }
  .btn-group button.active,
  .btn-group button:hover {
    background: #007acc;
    color: white;
  }
  .plot-container {
    margin-top: 1em;
  }
</style>

<div class="btn-group">
  <button data-target="m2-xik0.05" class="active">more aversion</button>
  <button data-target="m2-xik0.1">less aversion</button>
  <button data-target="m2-xik1e5">baseline</button>
</div>

<div id="m2-xik0.05" class="plot-container">
  <iframe src="../_static/df_V_m2_surface_xik0.05.html"
          width="100%" height="470" style="border:none;"></iframe>
</div>
<div id="m2-xik0.1" class="plot-container" style="display:none;">
  <iframe src="../_static/df_V_m2_surface_xik0.1.html"
          width="100%" height="470" style="border:none;"></iframe>
</div>
<div id="m2-xik1e5" class="plot-container" style="display:none;">
  <iframe src="../_static/df_V_m2_surface_xik1e+05.html"
          width="100%" height="470" style="border:none;"></iframe>
</div>

<script>
/* scope to m2 elements only */
(function() {
  const buttons = document.querySelectorAll('[data-target^="m2-"]');
  const paneIDs = ['m2-xik0.05', 'm2-xik0.1', 'm2-xik1e5'];

  buttons.forEach(btn => {
    btn.addEventListener('click', () => {
      /* toggle active class */
      buttons.forEach(b => b.classList.remove('active'));
      btn.classList.add('active');

      /* show / hide panes */
      const sel = btn.dataset.target;
      paneIDs.forEach(id => {
        document.getElementById(id).style.display =
          (id === sel ? 'block' : 'none');
      });
    });
  });
})();
</script>
